# Indicator-Refined Gauss-Bonnet Benchmark on a Biconcave Surface

This notebook reproduces the polynomial-degree convergence experiment for the biconcave algebraic surface. The workflow is intentionally separated into two stages:

1. Build a conforming indicator-refined reference mesh using `refine_by_indicator`.
2. On the adapted mesh, run a fixed polynomial-degree sweep with high-order simplex quadrature.


## Imports and Experiment Parameters

The notebook assumes it is executed from the `examples/` directory. The output figure is saved in the repository-level `images/` directory.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
from pathlib import Path
from time import time
from numba import njit

from surfgeopy import (
    SurfaceMesh,
    LevelSetSurface,
    IntegrationConfig,
    integrate,
    refine_by_indicator,
    PULL_BACK_GAUSS,
)

mesh_path = Path("../meshes/bioconcave_N=3144_d=0.5_c=0.375.mat")
output_path = Path("../images/G_bonnet_biconcave_indicator_refined_simplex.pdf")

adapt_steps = 6
threshold_fraction = 0.25
Nrange = list(range(2, 22))
exact_gauss_bonnet_value = 4.0 * np.pi


## Level Set and Gauss-Bonnet Integrand

The biconcave surface is represented as the zero level set of `phi`. The integrand `fun_1` is the Gaussian-curvature contribution used in the Gauss-Bonnet benchmark.


In [ ]:
d = 0.5
c = 0.375

@njit(fastmath=True)
def phi(x: np.ndarray):
    return (d**2 + x[0]**2 + x[1]**2 + x[2]**2)**3 - 8*d**2*(x[1]**2 + x[2]**2) - c**4

@njit(fastmath=True)
def dphi(x: np.ndarray):
    return np.array([
        6*x[0]*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2,
        6*x[1]*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2 - 16*d**2*x[1],
        6*x[2]*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2 - 16*d**2*x[2],
    ])

@njit(fastmath=True)
def fun_1(x: np.ndarray):
    return (6*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)* \
         ((-16*d**2*x[2] + 6*x[2]*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2)* \
            (24*x[0]**2*x[2]*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2* \
               (16*d**2 - 6*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2) - \
              24*x[1]*x[2]*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2* \
               (-16*d**2*x[1] + 6*x[1]*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2) + \
              (-16*d**2*x[2] + 6*x[2]*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2)* \
               (-96*x[0]**2*x[1]**2*(d**2 + x[0]**2 + x[1]**2 + x[2]**2) + \
                 (d**2 + 5*x[0]**2 + x[1]**2 + x[2]**2)* \
                  (-16*d**2 + 24*x[1]**2*(d**2 + x[0]**2 + x[1]**2 + x[2]**2) + \
                    6*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2))) + \
           (-16*d**2*x[1] + 6*x[1]*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2)* \
            (24*x[0]**2*x[1]*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2* \
               (16*d**2 - 6*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2) -  \
              24*x[1]*x[2]*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2* \
               (-16*d**2*x[2] + 6*x[2]*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2) + \
              (-16*d**2*x[1] + 6*x[1]*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2)* \
               (-96*x[0]**2*x[2]**2*(d**2 + x[0]**2 + x[1]**2 + x[2]**2) + \
                 (d**2 + 5*x[0]**2 + x[1]**2 + x[2]**2)* \
                  (-16*d**2 + 24*x[2]**2*(d**2 + x[0]**2 + x[1]**2 + x[2]**2) +  \
                    6*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2))) +  \
           6*x[0]**2*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2* \
            (4*x[1]*(16*d**2 - 6*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2)* \
               (-16*d**2*x[1] + 6*x[1]*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2) +  \
              4*x[2]*(16*d**2 - 6*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2)* \
               (-16*d**2*x[2] + 6*x[2]*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2) + \
              (d**2 + x[0]**2 + x[1]**2 + x[2]**2)* \
               (-576*x[1]**2*x[2]**2*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2 + \
                 (-16*d**2 + 24*x[1]**2*(d**2 + x[0]**2 + x[1]**2 + x[2]**2) + \
                    6*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2)* \
                  (-16*d**2 + 24*x[2]**2*(d**2 + x[0]**2 + x[1]**2 + x[2]**2) +  \
                    6*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2)))))/ \
       (36*x[0]**2*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**4 +  \
          (16*d**2*x[1] - 6*x[1]*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2)**2 +  \
          (16*d**2*x[2] - 6*x[2]*(d**2 + x[0]**2 + x[1]**2 + x[2]**2)**2)**2)**2


## Mesh Loading

The older MATLAB mesh files store vertices and faces in the convention used by the original scripts. The helper below converts them to the `SurfaceMesh` layout expected by the modern `surfgeopy` API.


In [ ]:
def load_old_mat_mesh(mesh_path: Path) -> SurfaceMesh:
    mesh_mat = scipy.io.loadmat(mesh_path)

    vertices = mesh_mat["xs"]
    faces = mesh_mat["surfs"] - 1

    if vertices.shape[0] == 3 and vertices.shape[1] != 3:
        vertices = vertices.T

    if faces.shape[0] in {3, 4} and faces.shape[1] != 3:
        faces = faces.T

    return SurfaceMesh(vertices, faces.astype(int))


mesh = load_old_mat_mesh(mesh_path)
surface = LevelSetSurface(mesh, phi, dphi)

print(f"Initial vertices: {surface.mesh.n_vertices}")
print(f"Initial faces:    {surface.mesh.n_faces}")


## Conforming Indicator Refinement

This is the mesh-adaptation stage. The indicator is evaluated at the affine center of each reference triangle, using

\[
\eta_T = |f(x_T)|,
\]

and a face is marked when `eta_T > threshold_fraction * eta_max`. The refinement is conforming: marked faces are red-refined, and neighboring faces with split edges are green-refined to avoid hanging edges.


In [ ]:
t0 = time()

adapted = refine_by_indicator(
    surface,
    fun_1,
    max_iterations=adapt_steps,
    threshold_fraction=threshold_fraction,
)

adapted_surface = adapted.final_surface
adaptation_time = time() - t0

print(adapted.summary())
for step in adapted.history:
    print(
        f"adapt step={step.iteration} | "
        f"faces={step.n_faces} | "
        f"marked={step.n_marked_faces} | "
        f"eta_max={step.max_indicator:.3e} | "
        f"threshold={step.threshold:.3e}"
    )

print(f"Adaptation time: {adaptation_time:.3f} s")


## Polynomial-Degree Study on the Adapted Mesh

Each run reconstructs the high-order curved surface on the same adapted reference mesh. The quadrature rule is the simplex-based pull-back Gaussian rule used by `surfgeopy` for the square-squeezing formulation.


In [ ]:
def gauss_bonnet_error(k: int) -> float:
    quadrature_degree = k + 1
    config = IntegrationConfig(
        interpolation_degree=int(k),
        refinement_level=0,
        integration_degree=quadrature_degree,
        quadrature_rule=PULL_BACK_GAUSS,
    )

    t0 = time()
    result = integrate(adapted_surface, fun_1, config)
    elapsed = time() - t0

    relative_error = abs((result.total - exact_gauss_bonnet_value) / exact_gauss_bonnet_value)

    print(
        f"k={k:2d} | "
        f"q={quadrature_degree:2d} | "
        f"integral={result.total:.16e} | "
        f"rel_error={relative_error:.3e} | "
        f"time={elapsed:.2f}s"
    )

    return relative_error


In [ ]:
error1 = []

for degree in Nrange:
    error1.append(gauss_bonnet_error(degree))


## Convergence Plot

The plot reports the relative Gauss-Bonnet error obtained after conforming indicator refinement of the reference mesh.


In [ ]:
plt.figure(figsize=(6, 4))
plt.semilogy(Nrange, error1, "-or", label=r"HOVE$_k$")
plt.xlabel("Polynomial degree", fontsize=13)
plt.ylabel("Relative error", fontsize=13)
plt.legend(prop={"size": 13}, frameon=False, loc="lower left")
plt.grid()
plt.xticks(np.arange(min(Nrange), max(Nrange) + 1, 1.0))
plt.ylim([1.0e-15, 1.0e-0])

output_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path, bbox_inches="tight")
plt.show()
